In [66]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
apoorvwatsky_bank_transaction_data_path = kagglehub.dataset_download('apoorvwatsky/bank-transaction-data')

print('Data source import complete.')


Data source import complete.


In [67]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/apoorvwatsky/bank-transaction-data/bank.xlsx


In [68]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "bank.xlsx"
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "apoorvwatsky/bank-transaction-data",
  file_path,
)
display("First 5 records:", df.head())

/tmp/ipykernel_55/3108690181.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


'First 5 records:'

,Account No,DATE,TRANSACTION DETAILS,CHQ.NO.,VALUE DATE,WITHDRAWAL AMT,DEPOSIT AMT,BALANCE AMT,.
0,409000611074',2017-06-29,TRF FROM Indiaforensic SERVICES,NaN,2017-06-29,NaN,1000000.0,1000000.0,.
1,409000611074',2017-07-05,TRF FROM Indiaforensic SERVICES,NaN,2017-07-05,NaN,1000000.0,2000000.0,.
2,409000611074',2017-07-18,FDRL/INTERNAL FUND TRANSFE,NaN,2017-07-18,NaN,500000.0,2500000.0,.
3,409000611074',2017-08-01,TRF FRM Indiaforensic SERVICES,NaN,2017-08-01,NaN,3000000.0,5500000.0,.
4,409000611074',2017-08-16,FDRL/INTERNAL FUND TRANSFE,NaN,2017-08-16,NaN,500000.0,6000000.0,.


In [69]:
def dataDescription(df):
    data = df.copy()
    
    print("\nShape of Data :", data.shape)
    print("\nData Types :\n", data.dtypes)
    print("\nData wise columns :", data.columns.tolist())
    print("\nFinding the null values :\n", data.isnull().sum())
    print("\nDuplicate Rows :", data.duplicated().sum())
    
    print("\nColumn wise unique values :")
    for col in data.columns:
        print(f"{col} unique values :", data[col].nunique())
dataDescription(df)


Shape of Data : (116201, 9)

Data Types :
 Account No                     object
DATE                   datetime64[ns]
TRANSACTION DETAILS            object
CHQ.NO.                       float64
VALUE DATE             datetime64[ns]
WITHDRAWAL AMT                float64
DEPOSIT AMT                   float64
BALANCE AMT                   float64
.                              object
dtype: object

Data wise columns : ['Account No', 'DATE', 'TRANSACTION DETAILS', 'CHQ.NO.', 'VALUE DATE', 'WITHDRAWAL AMT', 'DEPOSIT AMT', 'BALANCE AMT', '.']

Finding the null values :
 Account No                  0
DATE                        0
TRANSACTION DETAILS      2499
CHQ.NO.                115296
VALUE DATE                  0
WITHDRAWAL AMT          62652
DEPOSIT AMT             53549
BALANCE AMT                 0
.                           0
dtype: int64

Duplicate Rows : 39

Column wise unique values :
Account No unique values : 10
DATE unique values : 1294
TRANSACTION DETAILS unique values : 44

In [70]:
def dataCleaning(df):
    data = df.copy()

    cols_to_drop = ['.', 'CHQ.NO.']
    data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True, errors='ignore')

    data['WITHDRAWAL AMT'] = data['WITHDRAWAL AMT'].fillna(0)
    data['DEPOSIT AMT'] = data['DEPOSIT AMT'].fillna(0)

    data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')
    data['VALUE DATE'] = pd.to_datetime(data['VALUE DATE'], errors='coerce')

    data.sort_values(['Account No', 'DATE'], inplace=True)

    data['net_flow'] = data['DEPOSIT AMT'] - data['WITHDRAWAL AMT']

    daily_df = (
        data.groupby(['Account No', 'DATE'], as_index=False)
        .agg(
            WITHDRAWAL_AMT=('WITHDRAWAL AMT', 'sum'),
            DEPOSIT_AMT=('DEPOSIT AMT', 'sum'),
            net_flow=('net_flow', 'sum'),
            BALANCE_AMT=('BALANCE AMT', 'last'),
            txn_count=('TRANSACTION DETAILS', 'count')
        )
    )

    daily_df.sort_values(['Account No', 'DATE'], inplace=True)
    daily_df.reset_index(drop=True, inplace=True)
    
    return daily_df
cleandata = dataCleaning(df)
display(cleandata)
cleandata.columns

,Account No,DATE,WITHDRAWAL_AMT,DEPOSIT_AMT,net_flow,BALANCE_AMT,txn_count
0,1196428',2015-01-01,0.00,2000000.0,2000000.00,-1.584916e+09,2
1,1196428',2015-01-02,5500056.18,2462620.0,-3037436.18,-1.587953e+09,9
2,1196428',2015-01-03,2200000.00,2948160.0,748160.00,-1.587205e+09,4
3,1196428',2015-01-05,3231000.00,3877320.0,646320.00,-1.586559e+09,5
4,1196428',2015-01-06,3880056.18,3650750.0,-229306.18,-1.586788e+09,6
...,...,...,...,...,...,...,...
6213,409000611074',2019-01-29,301645.00,300000.0,-1645.00,1.525434e+06,3
6214,409000611074',2019-01-30,404412.00,0.0,-404412.00,1.121022e+06,2
6215,409000611074',2019-01-31,225251.00,300000.0,74749.00,1.195771e+06,3
6216,409000611074',2019-02-01,133571.00,0.0,-133571.00,1.062200e+06,2


Index(['Account No', 'DATE', 'WITHDRAWAL_AMT', 'DEPOSIT_AMT', 'net_flow',
       'BALANCE_AMT', 'txn_count'],
      dtype='object')

In [71]:
def calculate_slope(series):
    y = series.to_numpy()
    
    if len(y) < 2:
        return np.nan
    
    x = np.arange(len(y))
    return np.polyfit(x, y, 1)[0]

In [72]:
def featureCreation(df):
    data = df.copy()
    data.sort_values(['Account No', 'DATE'], inplace=True)

    data['net_flow'] = data['DEPOSIT_AMT'] - data['WITHDRAWAL_AMT']

    data['days_since_last_txn'] = (
        data.groupby('Account No')['DATE'].diff().dt.days
    )

    data['rolling_txn_7d'] = (
        data.groupby('Account No')['txn_count']
        .transform(lambda x: x.rolling(7, min_periods=1).sum())
    )

    data['rolling_dep_30d'] = (
        data.groupby('Account No')['DEPOSIT_AMT']
        .transform(lambda x: x.rolling(30, min_periods=1).sum())
    )

    data['rolling_wdr_30d'] = (
        data.groupby('Account No')['WITHDRAWAL_AMT']
        .transform(lambda x: x.rolling(30, min_periods=1).sum())
    )

    data['rolling_net_30d'] = (
        data.groupby('Account No')['net_flow']
        .transform(lambda x: x.rolling(30, min_periods=1).sum())
    )

    data['balance_std_30d'] = (
        data.groupby('Account No')['BALANCE_AMT']
        .transform(lambda x: x.rolling(30, min_periods=2).std())
    )

    data['txn_std_30d'] = (
        data.groupby('Account No')['net_flow']
        .transform(lambda x: x.rolling(30, min_periods=2).std())
    )

    data['balance_ma_7d'] = (
        data.groupby('Account No')['BALANCE_AMT']
        .transform(lambda x: x.rolling(7, min_periods=1).mean())
    )

    data['wd_to_dep_ratio_30d'] = (
        data['rolling_wdr_30d'] / (data['rolling_dep_30d'] + 1e-6)
    )

    data['day_of_week'] = data['DATE'].dt.dayofweek
    data['month'] = data['DATE'].dt.month

    data["balance_slope_30d"] = (
        data.groupby("Account No")["BALANCE_AMT"]
        .transform(lambda x: x.rolling(30, min_periods=2)
                   .apply(calculate_slope, raw=False))
    )

    data["withdrawal_acceleration_30d"] = (
        data.groupby("Account No")["rolling_wdr_30d"].diff(7)
    )

    data["deposit_acceleration_30d"] = (
        data.groupby("Account No")["rolling_dep_30d"].diff(7)
    )

    data["txn_acceleration_7d"] = (
        data.groupby("Account No")["rolling_txn_7d"].diff(3)
    )

    data["cash_buffer_ratio"] = (
        data["balance_ma_7d"] / (data["rolling_wdr_30d"] + 1e-6)
    )

    data["flow_volatility_ratio"] = (
        data["balance_std_30d"] / (data["balance_ma_7d"] + 1e-6)
    )

    data["deposit_std_30d"] = (
        data.groupby("Account No")["DEPOSIT_AMT"]
        .transform(lambda x: x.rolling(30, min_periods=2).std())
    )

    data["deposit_consistency"] = (
        data["rolling_dep_30d"] / (data["deposit_std_30d"] + 1e-6)
    )

    data["future_withdrawal_7d"] = (
        data.groupby("Account No")["WITHDRAWAL_AMT"]
        .transform(lambda x: x.shift(-1).rolling(7, min_periods=1).sum())
    )

    data["future_deposit_30d"] = (
        data.groupby("Account No")["DEPOSIT_AMT"]
        .transform(lambda x: x.shift(-1).rolling(30, min_periods=1).sum())
    )

    data["min_balance_next_30d"] = (
        data.groupby("Account No")["BALANCE_AMT"]
        .transform(lambda x: x.shift(-1).rolling(30, min_periods=1).min())
    )

    data["overdraft_30d_label"] = (
        data["min_balance_next_30d"] < 0
    ).astype(int)

    data["withdrawal_spike_score"] = (
        (data["WITHDRAWAL_AMT"] - (data["rolling_wdr_30d"] / 30)) /
        ((data["rolling_wdr_30d"] / 30) + 1e-6)
    )

    data["deposit_spike_score"] = (
        (data["DEPOSIT_AMT"] - (data["rolling_dep_30d"] / 30)) /
        ((data["rolling_dep_30d"] / 30) + 1e-6)
    )

    data["withdrawal_zscore_30d"] = (
        (data["WITHDRAWAL_AMT"] - (data["rolling_wdr_30d"] / 30)) /
        (data["txn_std_30d"] + 1e-6)
    )

    data["deposit_zscore_30d"] = (
        (data["DEPOSIT_AMT"] - (data["rolling_dep_30d"] / 30)) /
        (data["deposit_std_30d"] + 1e-6)
    )

    data["balance_drop_ratio"] = (
        (data["BALANCE_AMT"] - data["balance_ma_7d"]) /
        (data["balance_ma_7d"] + 1e-6)
    )

    data["withdrawal_pressure"] = (
        data["rolling_wdr_30d"] /
        (data["balance_ma_7d"] + 1e-6)
    )

    data["negative_flow_ratio"] = (
        data["rolling_net_30d"] /
        (data["balance_ma_7d"] + 1e-6)
    )

    data["stability_score"] = (
        1 / (data["balance_std_30d"] + data["txn_std_30d"] + 1e-6)
    )

    data["large_withdrawal_flag"] = (
        data["WITHDRAWAL_AMT"] >
        (data["rolling_wdr_30d"] / 10)
    ).astype(int)

    data["large_deposit_flag"] = (
        data["DEPOSIT_AMT"] >
        (data["rolling_dep_30d"] / 10)
    ).astype(int)

    data['has_overdraft'] = data['overdraft_30d_label']
    
    data.dropna(inplace=True)

    return data
processed_df = featureCreation(cleandata)
display(processed_df.shape)
processed_df.columns

(6148, 41)

Index(['Account No', 'DATE', 'WITHDRAWAL_AMT', 'DEPOSIT_AMT', 'net_flow',
       'BALANCE_AMT', 'txn_count', 'days_since_last_txn', 'rolling_txn_7d',
       'rolling_dep_30d', 'rolling_wdr_30d', 'rolling_net_30d',
       'balance_std_30d', 'txn_std_30d', 'balance_ma_7d',
       'wd_to_dep_ratio_30d', 'day_of_week', 'month', 'balance_slope_30d',
       'withdrawal_acceleration_30d', 'deposit_acceleration_30d',
       'txn_acceleration_7d', 'cash_buffer_ratio', 'flow_volatility_ratio',
       'deposit_std_30d', 'deposit_consistency', 'future_withdrawal_7d',
       'future_deposit_30d', 'min_balance_next_30d', 'overdraft_30d_label',
       'withdrawal_spike_score', 'deposit_spike_score',
       'withdrawal_zscore_30d', 'deposit_zscore_30d', 'balance_drop_ratio',
       'withdrawal_pressure', 'negative_flow_ratio', 'stability_score',
       'large_withdrawal_flag', 'large_deposit_flag', 'has_overdraft'],
      dtype='object')

In [73]:
processed_df["future_net_flow"] = (
    processed_df["future_deposit_30d"] - processed_df["future_withdrawal_7d"]
)
processed_df["future_txn_volume"] = (
    processed_df.groupby("Account No")["txn_count"]
    .shift(-7)
)

In [74]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    accuracy_score,
    classification_report
)

from xgboost import XGBRegressor, XGBClassifier

In [75]:
# ===========================
# TARGET ENGINEERING BLOCK
# ===========================

# ---- Forecast Derived Targets ----

processed_df["future_net_flow"] = (
    processed_df["future_deposit_30d"] - processed_df["future_withdrawal_7d"]
)

processed_df["future_txn_volume"] = (
    processed_df["rolling_txn_7d"].shift(-7)
)

# ---- Liquidity Targets ----

processed_df["liquidity_stress_risk"] = (
    (processed_df["cash_buffer_ratio"] < 0.2).astype(int)
)

processed_df["negative_balance_persistence"] = (
    (processed_df["BALANCE_AMT"] < 0)
    .groupby(processed_df["Account No"])
    .rolling(5)
    .sum()
    .reset_index(level=0, drop=True)
    > 3
).astype(int)

processed_df["cash_flow_imbalance_risk"] = (
    (processed_df["negative_flow_ratio"] > 0.6).astype(int)
)

# ---- Behavior Change Targets ----

processed_df["sudden_deposit_drop"] = (
    (processed_df["deposit_acceleration_30d"] < -0.5).astype(int)
)

processed_df["sudden_withdrawal_surge"] = (
    (processed_df["withdrawal_acceleration_30d"] > 0.5).astype(int)
)

processed_df["inactivity_prediction"] = (
    (processed_df["days_since_last_txn"] > 15).astype(int)
)

processed_df["transaction_silence_detection"] = (
    (processed_df["txn_count"] == 0).astype(int)
)

# ---- Fraud Signals ----

processed_df["velocity_fraud_flag"] = (
    (processed_df["txn_acceleration_7d"] > 2).astype(int)
)

processed_df["rapid_balance_drain_alert"] = (
    (processed_df["balance_drop_ratio"] > 0.4).astype(int)
)

processed_df["round_amount_fraud_pattern"] = (
    (processed_df["WITHDRAWAL_AMT"] % 1000 == 0).astype(int)
)

processed_df["micro_transaction_burst"] = (
    (processed_df["txn_count"] > processed_df["rolling_txn_7d"] * 2).astype(int)
)

# ---- Account Health ----

processed_df["volatile_account_classification"] = (
    (processed_df["balance_std_30d"] > processed_df["balance_std_30d"].median()).astype(int)
)

processed_df["financial_health_index"] = (
    processed_df["stability_score"] *
    (1 - processed_df["negative_flow_ratio"]) *
    processed_df["cash_buffer_ratio"]
)

processed_df["high_risk_lifestyle_pattern"] = (
    (processed_df["withdrawal_pressure"] > 0.7).astype(int)
)

In [77]:
# ==========================================================
# FULL MULTI-MODEL TRAINING PIPELINE (REG + CLASSIFICATION)
# ==========================================================

import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    accuracy_score
)

from xgboost import XGBRegressor, XGBClassifier


# -----------------------------
# TIME-BASED SPLIT
# -----------------------------
def time_based_split(df, target, split_ratio=0.8):

    train_idx = []
    val_idx = []

    for acc, acc_df in df.groupby("Account No"):
        acc_df = acc_df.sort_values("DATE")
        split_point = int(len(acc_df) * split_ratio)

        train_idx.extend(acc_df.index[:split_point])
        val_idx.extend(acc_df.index[split_point:])

    X_train = df.loc[train_idx, predictive_cols]
    X_val = df.loc[val_idx, predictive_cols]

    y_train = target.loc[train_idx]
    y_val = target.loc[val_idx]

    return X_train, X_val, y_train, y_val


# -----------------------------
# TARGET CONFIG
# -----------------------------
MODEL_CONFIG = {

    # 🔵 Regression
    "future_withdrawal_7d": {"type": "regression"},
    "future_deposit_30d": {"type": "regression"},
    "future_net_flow": {"type": "regression"},
    "min_balance_next_30d": {"type": "regression"},
    "future_txn_volume": {"type": "regression"},
    "financial_health_index": {"type": "regression"},

    # 🔴 Classification
    "overdraft_30d_label": {"type": "classification"},
    "liquidity_stress_risk": {"type": "classification"},
    "negative_balance_persistence": {"type": "classification"},
    "cash_flow_imbalance_risk": {"type": "classification"},
    "sudden_deposit_drop": {"type": "classification"},
    "sudden_withdrawal_surge": {"type": "classification"},
    "inactivity_prediction": {"type": "classification"},
    "transaction_silence_detection": {"type": "classification"},
    "velocity_fraud_flag": {"type": "classification"},
    "rapid_balance_drain_alert": {"type": "classification"},
    "round_amount_fraud_pattern": {"type": "classification"},
    "micro_transaction_burst": {"type": "classification"},
    "large_withdrawal_flag": {"type": "classification"},
    "large_deposit_flag": {"type": "classification"},
    "volatile_account_classification": {"type": "classification"},
    "high_risk_lifestyle_pattern": {"type": "classification"}
}


# -----------------------------
# PARAM GRID
# -----------------------------
param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9]
}


# -----------------------------
# TRAINING LOOP
# -----------------------------
trained_models = {}
model_metrics = {}

for target_name, config in MODEL_CONFIG.items():

    print(f"\n==============================")
    print(f"Training: {target_name}")
    print("==============================")

    if target_name not in processed_df.columns:
        print("Target not found. Skipping.")
        continue

    y = processed_df[target_name]

    # Skip if target has only 1 class
    if config["type"] == "classification":
        if y.nunique() < 2:
            print("Only one class present. Skipping.")
            continue

    X_train, X_val, y_train, y_val = time_based_split(
        processed_df,
        y
    )

    if config["type"] == "regression":

        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        )
        scoring_metric = "neg_root_mean_squared_error"

    else:

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42
        )
        scoring_metric = "roc_auc"

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter=15,
        scoring=scoring_metric,
        cv=3,
        verbose=1,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    best_pipeline = search.best_estimator_
    trained_models[target_name] = best_pipeline

    # Validation Evaluation
    y_pred = best_pipeline.predict(X_val)

    if config["type"] == "regression":

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)

        print("RMSE:", rmse)
        print("MAE :", mae)
        print("R2  :", r2)

        model_metrics[target_name] = {
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        }

    else:

        y_prob = best_pipeline.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_prob)
        acc = accuracy_score(y_val, y_pred)

        print("ROC-AUC:", roc)
        print("Accuracy:", acc)

        model_metrics[target_name] = {
            "ROC-AUC": roc,
            "Accuracy": acc
        }


# -----------------------------
# SAVE ALL MODELS
# -----------------------------
joblib.dump(trained_models, "banking_all_models.pkl")
joblib.dump(model_metrics, "banking_model_metrics.pkl")

print("\nAll models trained and saved successfully.")


Training: future_withdrawal_7d
Fitting 3 folds for each of 15 candidates, totalling 45 fits
RMSE: 42102243.186197385
MAE : 22820909.596780792
R2  : 0.859272122270525

Training: future_deposit_30d
Fitting 3 folds for each of 15 candidates, totalling 45 fits
RMSE: 43503629.42618631
MAE : 23619605.376140602
R2  : 0.991003750920383

Training: future_net_flow
Fitting 3 folds for each of 15 candidates, totalling 45 fits
RMSE: 54456525.467347756
MAE : 27651708.722285252
R2  : 0.9774428243604347

Training: min_balance_next_30d
Fitting 3 folds for each of 15 candidates, totalling 45 fits
RMSE: 46668102.586085625
MAE : 20656975.757000037
R2  : 0.9961477138336727

Training: future_txn_volume
Fitting 3 folds for each of 15 candidates, totalling 45 fits


ValueError: Input contains NaN.